# Stage 1: combine embedding views

This tutorial uses **small synthetic arrays** to demonstrate the API. They are not biological results. The real LINCS experiments are run on the cluster using the accompanying Slurm script. Fusion combines representations of the same entities; it does not infer correspondences between unrelated genes, compounds and cells.

In [1]:
import numpy as np
import pandas as pd
from embpy.io import EmbeddingProvenance, EmbeddingResult, to_anndata
from embpy.tl.fusion import (ConcatFusion, PCAFusion, subset_views,
                            fuse_scores, fuse_ranks, views_from_anndata)
from embpy.tl import benchmark_fusion, benchmark_fusion_retrieval

rng = np.random.default_rng(7)
ids = tuple(f"example_{i:03d}" for i in range(60))
latent = rng.normal(size=(60, 5))
a = latent @ rng.normal(size=(5, 12)) + rng.normal(scale=0.2, size=(60, 12))
b = latent @ rng.normal(size=(5, 8)) + rng.normal(scale=0.2, size=(60, 8))
views = {
    "view_a": EmbeddingResult(a, ids, "gene", "synthetic_id", EmbeddingProvenance("synthetic_a")),
    # Deliberately reverse rows: alignment uses IDs, not positional coincidence.
    "view_b": EmbeddingResult(b[::-1], ids[::-1], "gene", "synthetic_id", EmbeddingProvenance("synthetic_b")),
}

## Split before fitting

The fitted operator stores feature statistics and its PCA basis. New entities can be transformed without refitting. In a real dataset, make this split at the compound/gene/donor level required by the scientific question.

In [2]:
train_views = subset_views(views, ids[:40])
test_views = subset_views(views, ids[40:])
fusion = PCAFusion(n_components=6, normalization="l2", random_state=0)
fusion.fit(train_views)
fused_train = fusion.transform(train_views)
fused_test = fusion.transform(test_views)
assert fused_test.entity_ids == ids[40:]
assert fused_test.matrix.shape == (20, 6)
print("Train / test shapes:", fused_train.matrix.shape, fused_test.matrix.shape)
print("Coverage:", fusion.coverage_)

Train / test shapes: (40, 6) (20, 6)
Coverage: {'join': 'strict', 'n_entities': 40, 'input_counts': {'view_a': 40, 'view_b': 40}, 'dropped_counts': {'view_a': 0, 'view_b': 0}}


## Normalization, block balance and weights

Z-scoring is per feature; L2 normalization is per entity within a view. Weights are fixed and multiply blocks by their square root. Optional block balancing divides by the training RMS row norm. Matching widths using independent PCAs does not make their coordinates safe to average.

In [3]:
concat = ConcatFusion(
    normalization={"view_a": "zscore", "view_b": "l2"},
    balance_blocks=True,
    weights={"view_a": 2.0, "view_b": 1.0},
)
joined = concat.fit_transform(train_views)
assert joined.matrix.shape == (40, 20)
print(joined.provenance.extra["fusion"]["view_order"])

['view_a', 'view_b']


## Export and retain fitted state

Outputs use the existing embedding contract. Save the fitted operator if you need to transform new data later. Load only trusted joblib files.

In [4]:
import tempfile
from pathlib import Path
import joblib

adata = to_anndata(fused_test)
# Gene entities use var/varm in standalone AnnData exports.
assert adata.n_vars == 20
assert tuple(adata.var_names) == fused_test.entity_ids
with tempfile.TemporaryDirectory() as directory:
    path = Path(directory) / "fusion.joblib"
    joblib.dump(fusion, path)
    restored = joblib.load(path)
    np.testing.assert_allclose(restored.transform(test_views).matrix, fused_test.matrix)
print("Fitted-state roundtrip passed")

Fitted-state roundtrip passed


## Score and rank ensembles

Late fusion combines query-to-candidate evidence rather than coordinates. Every view must use the same candidate universe; row and column order can differ. Larger fused output is always better. Here the queries are held-out synthetic entities and the candidates are the training entities.

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

scores = {
    name: pd.DataFrame(cosine_similarity(test_views[name].matrix, train_views[name].matrix),
                       index=ids[40:], columns=ids[:40])
    for name in views
}
mean_scores = fuse_scores(scores)
mean_ranks = fuse_ranks(scores)
assert mean_scores.shape == mean_ranks.shape == (20, 40)
print("Best candidate for first query:", mean_ranks.iloc[0].idxmax())

Best candidate for first query: example_009


## Paired evaluation

This is a tiny synthetic execution check, not a performance claim. Every method uses the same grouped folds. Production benchmarks use nested tuning (`tune=True`) and run on the cluster. `single_n_components` can match the output width of a PCA comparator; full concatenation has more features and therefore a different readout capacity.

In [6]:
targets = pd.Series(latent[:, 0] + 0.5 * latent[:, 1], index=ids)
groups = pd.Series([f"group_{i // 2}" for i in range(60)], index=ids)
operators = {"concat_l2": ConcatFusion(normalization="l2"),
             "pca6": PCAFusion(n_components=6)}
comparison = benchmark_fusion(views, targets, groups=groups, fusions=operators,
                              cv=3, seeds=(0,), tune=False)
for _, split in comparison.splits.groupby(["seed", "fold"]):
    assert set(split.loc[split.partition == "train", "group"]).isdisjoint(
        split.loc[split.partition == "test", "group"])
print(comparison.scores.groupby("configuration")[["mse", "pearson", "output_dim"]].mean())

                    mse   pearson  output_dim
configuration                                
concat_l2      0.183707  0.907374        20.0
pca6           0.182556  0.909896         6.0
single:view_a  0.188502  0.903981        12.0
single:view_b  0.221426  0.897236         8.0


Retrieval relevance is exact equality of categorical labels. A query without any relevant training candidate has undefined recall; its zero precision/hit remains included and coverage is reported.

In [7]:
labels = pd.Series(np.where(latent[:, 0] > 0, "positive", "negative"), index=ids)
retrieval = benchmark_fusion_retrieval(views, labels, groups=groups, fusions=operators,
                                      cv=3, seeds=(0,), k=5)
print(retrieval.scores.groupby("configuration")[["precision_at_k", "recall_at_k"]].mean())

               precision_at_k  recall_at_k
configuration                             
concat_l2            0.730000     0.181328
pca6                 0.743333     0.185460
rank_mean            0.706667     0.175509
score_mean           0.730000     0.181328
single:view_a        0.730000     0.181473
single:view_b        0.720000     0.178970


## Using AnnData and running the real benchmark

Use `views_from_anndata(adata, ["X_model_a", "X_model_b"])` to adapt existing `.obsm` matrices. Observation IDs must be unique; repeated compound names belong in `groups` or `benchmark_embeddings(..., groupby=...)`.

The cluster entry point is `python -m embpy.scripts.benchmark_fusion_lincs`. Submit `submission_scripts/run_fusion_lincs.sbatch` with `EMBPY_PYTHON` and `EMBPY_SOURCE_ROOT` set, plus `--input-h5ad`, `--embedding-root`, and `--output-dir`. The [fusion guide](../fusion.md) explains the audited feature columns, regenerated classical features, structure-level aggregation, group splits and output files.

Fold standard deviations are descriptive, not confidence intervals. Use paired entity/group-level results for uncertainty, and a separate final holdout when selecting a winning fusion configuration.